# SkillPulse data exploration

This notebook explores the included job-posting sample and its ground-truth labels. The bundled dataset is **synthetic**, generated from explicit skill-demand curves with noise and deliberately varied posting text. It is useful for checking the pipeline and learning the data shape, but it is not evidence about the real labor market.

Run cells from top to bottom. No API or database is required.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "sample" / "job_postings.csv").is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Could not find data/sample/job_postings.csv; open this notebook inside the SkillPulse project.")
    ROOT = ROOT.parent

SAMPLE_DIR = ROOT / "data" / "sample"
jobs = pd.read_csv(SAMPLE_DIR / "job_postings.csv", low_memory=False)
ground_truth = pd.read_csv(SAMPLE_DIR / "job_postings_ground_truth.csv")
jobs["posted_date"] = pd.to_datetime(jobs["posted_date"], errors="coerce")
print(f"Repository: {ROOT}")
print(f"Posting rows: {len(jobs):,}; ground-truth rows: {len(ground_truth):,}")

## Shape and data quality

Inspect the input fields and missingness before deriving any metrics. Duplicate source IDs and unparsable dates are useful checks when changing the generator or testing ingestion.

In [ ]:
display(jobs.head(5))
display(pd.DataFrame({
    "dtype": jobs.dtypes.astype(str),
    "missing": jobs.isna().sum(),
    "missing_pct": (jobs.isna().mean() * 100).round(2),
}))
print(f"Duplicate job IDs: {jobs['job_id'].duplicated().sum():,}")
print(f"Dates that could not be parsed: {jobs['posted_date'].isna().sum():,}")
print(f"Unique job IDs: {jobs['job_id'].nunique():,}")

## Coverage and posting characteristics

Yearly volume and description lengths provide context for the observations. The API's demand metric is the share of postings that mention a skill, not raw mention count.

In [ ]:
dated = jobs.dropna(subset=["posted_date"]).copy()
dated["year"] = dated["posted_date"].dt.year
postings_by_year = dated.groupby("year").size().rename("postings").to_frame()
display(postings_by_year)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
postings_by_year["postings"].plot(kind="bar", ax=axes[0], color="#4f46e5", title="Postings by year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Postings")
axes[0].tick_params(axis="x", rotation=45)
description_lengths = jobs["description"].fillna("").str.len()
description_lengths.plot(kind="hist", bins=30, ax=axes[1], color="#0891b2", title="Description length")
axes[1].set_xlabel("Characters")
plt.tight_layout()
plt.show()
display(description_lengths.describe().to_frame("description_characters"))

In [ ]:
for column in ("industry", "source"):
    if column in jobs.columns:
        print(f"Top {column} values")
        display(jobs[column].fillna("(missing)").value_counts().head(12).rename("postings").to_frame())

print("Ground-truth roles")
display(ground_truth["role"].value_counts().head(12).rename("postings").to_frame())

## Ground-truth skill frequencies

The ground-truth file stores skills as pipe-delimited labels. This view is useful for understanding the generated benchmark; it is distinct from the skills that the production extraction pipeline detects from descriptions.

In [ ]:
skill_counts = (
    ground_truth["skills"].fillna("").str.split("|").explode().str.strip()
    .loc[lambda values: values.ne("")].value_counts().rename_axis("skill").rename("ground_truth_postings")
)
display(skill_counts.head(25).to_frame())
skill_counts.head(20).sort_values().plot(kind="barh", figsize=(9, 7), color="#16a34a", title="Most common ground-truth skills")
plt.xlabel("Postings mentioning skill")
plt.tight_layout()
plt.show()

## Interpretation

- These plots describe a generated test fixture; do not generalize them to actual hiring markets.
- Ground-truth labels are used to evaluate extraction, while dashboard analysis is computed from posting text and extracted skill links.
- Skill mention share is sensitive to the time window, geography, role mix and quality of the collected postings.